This notebook compares two LoRA fine-tunes of `Qwen3-1.7B-Base` on the Winogrande dataset: one trained with Adam, the other with IVON. We demonstrate that LoRA adapters trained with IVON achieve both better accuracy and better calibration than Adam. Besides, one can sample from the learned posterior and average the predictions, which further improves the calibration.

In [ ]:
import random
from contextlib import contextmanager
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, DataCollatorWithPadding
from peft import AutoPeftModelForCausalLM

# Plotting style
for _f in (Path.home() / ".fonts").glob("*.otf"):
    fm.fontManager.addfont(str(_f))

%config InlineBackend.figure_format = "svg"

plt.rcParams.update(
    {
        'font.family': 'serif',
        'font.serif': ['Times', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
        'axes.labelsize': 8,
        'font.size': 8,
        'legend.fontsize': 7,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.edgecolor': 'gray',
        'xtick.color': 'gray',
        'ytick.color': 'gray',
        'xtick.major.size': 2,
        'xtick.major.width': 0.5,
        'ytick.major.size': 2,
        'ytick.major.width': 0.5,
        'xtick.minor.size': 1,
        'xtick.minor.width': 0.5,
        'ytick.minor.size': 1,
        'ytick.minor.width': 0.5,
    }
)

In [ ]:
device = torch.device('cpu')

In [ ]:
# Hyperparameters
BASE_MODEL = "Qwen/Qwen3-1.7B-Base"
TASK       = "winogrande_s"
SEED       = 42
BATCH_SIZE = 128     # evaluation batch size
ESS        = 1e6     # IVON's effective sample size: larger for more concentrated posterior (less weight noise)
N_SAMPLES  = 16      # number of samples to draw from the learned posterior for prediction averaging

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Load tokenizer and models (in 8-bit quantization)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.bos_token

bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)

# Load with LoRA weights trained on the Winogrande-S set, with either Adam or IVON.
model_adam = AutoPeftModelForCausalLM.from_pretrained(
    "ckpts/adam", quantization_config=bnb_cfg, is_trainable=True
)
model_ivon = AutoPeftModelForCausalLM.from_pretrained(
    "ckpts/ivon", quantization_config=bnb_cfg, is_trainable=True
)

WinoGrande is a commonsense reasoning benchmark designed to test whether AI systems can resolve ambiguous references using commonsense knowledge. We used the `winogrande_s` subset to train the LoRA adapters which we just loaded, and we will evaluate the adapters on its validation set. Here we load the validation set and show an example:

In [ ]:
# Load the Winogrande validation set
raw = load_dataset("allenai/winogrande", TASK, split="validation")

def preprocess(examples):
    texts = [
        f"Select one of the choices that answers the following question: {s} "
        f"Choices: A. {o1}. B. {o2}. Answer:"
        for s, o1, o2 in zip(examples["sentence"], examples["option1"], examples["option2"])
    ]
    result = tokenizer(texts)
    result["labels"] = [{"1": 0, "2": 1, "": None}[a] for a in examples["answer"]]
    return result

processed = raw.map(
    preprocess, batched=True, remove_columns=raw.column_names, desc="Tokenizing"
)

collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def gpu_collator(batch):
    return {k: v.to(device) for k, v in collator(batch).items()}

val_loader = DataLoader(processed, collate_fn=gpu_collator, batch_size=BATCH_SIZE)
print("Val dataset size:", len(val_loader.dataset))

# Show one entry before vs. after preprocessing.
proc_row = processed[0]
print("\n[before] raw row:")
for k, v in raw[0].items():
    print(f"    {k:8s}= {v!r}")
print("\n[after]  tokenized entry:")
print(f"    decoded   = {tokenizer.decode(proc_row['input_ids'])!r}")
print(f"    input_ids = {proc_row['input_ids']}")
print(f"    labels    = {proc_row['labels']}")

Let's check the predictions of the two models on the first few examples:

In [ ]:
# Wrap the model to only return the logits for the last token and the two answer choices
class WrappedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.id_list = [tokenizer.encode(c, add_special_tokens=False)[0] for c in ("A", "B")]
        self.model = model

    def forward(self, *args, **kwargs):
        if len(args) > 0:
            assert (len(kwargs) == 0) and (len(args) == 1)
            kwargs = args[0]
        labels = kwargs.pop("labels", None)
        logits = self.model(**kwargs)["logits"]
        kwargs["labels"] = labels
        return logits[:, -1, self.id_list]

model_adam = WrappedModel(model_adam.to(device)).eval()
model_ivon = WrappedModel(model_ivon.to(device)).eval()

batch = next(iter(val_loader))
with torch.inference_mode():
    p_adam = torch.softmax(model_adam(**batch), dim=-1)
    p_ivon = torch.softmax(model_ivon(**batch), dim=-1)

for i in range(5):
    ex, gold = raw[i], batch["labels"][i].item()
    print(f"\nQ: {ex['sentence']}")
    print(f"   A. {ex['option1']}   B. {ex['option2']}   (gold = {'AB'[gold]})")
    for name, p in [("Adam", p_adam), ("IVON@mean", p_ivon)]:
        pA, pB = p[i, 0].item(), p[i, 1].item()
        pred = 0 if pA >= pB else 1
        print(f"   {name:10s} P(A)={pA:.3f}  P(B)={pB:.3f}  -> {'AB'[pred]}  {'✓' if pred == gold else '✗'}")


Then let's evaluate the two models on the entire validation set and compute the metrics (accuracy, ECE, NLL and Brier score):

- **ECE** (Expected Calibration Error): Predictions are grouped into confidence bins, and ECE is the average gap between confidence and actual accuracy across those bins. A model whose 80%-confident predictions are correct ~80% of the time would have an ECE near 0.
- **Brier score** is the squared error between the predicted probability vector and the one-hot label, averaged over examples. It penalises both wrong predictions and over-/under-confident ones.

(Lower is better for all metrics except accuracy.)

In [ ]:
# Helper functions for evaluation
@torch.inference_mode()
def calculate_metrics(probs: torch.Tensor, labels: torch.Tensor, num_bins: int = 10):
    preds = probs.argmax(dim=1)
    conf = probs.max(dim=1).values
    correct = (preds == labels).float()

    acc = correct.mean()
    nll = F.nll_loss(probs.log(), labels)
    one_hot = F.one_hot(labels, num_classes=probs.size(1))
    brier = (probs - one_hot).pow(2).sum(dim=1).mean()

    edges = torch.linspace(0.5, 1, num_bins + 1, device=probs.device)
    ece = torch.zeros((), device=probs.device)
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = ((conf >= lo) if i == 0 else (conf > lo)) & (conf <= hi)
        if mask.any():
            ece += mask.float().mean() * (correct[mask].mean() - conf[mask].mean()).abs()

    return acc.item(), nll.item(), ece.item(), brier.item()

@torch.inference_mode()
def mean_predict(model, loader):
    probs, labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        probs.append(torch.softmax(model(**batch), dim=-1))
        labels.append(batch["labels"])
    return torch.cat(probs).float(), torch.cat(labels)

def metrics_row(name, probs, labels):
    acc, nll, ece, brier = calculate_metrics(probs, labels)
    return {"model": name, "accuracy": acc, "nll": nll, "ece": ece, "brier": brier}

# Calculate and print metrics for both models
probs_adam, labels = mean_predict(model_adam, val_loader)
probs_ivon, _      = mean_predict(model_ivon, val_loader)
print(pd.DataFrame([
    metrics_row("Adam", probs_adam, labels),
    metrics_row("IVON@mean", probs_ivon, labels),
]).set_index("model").round(4))

Let's visualize the reliability diagrams of the two models. A reliability diagram is the graphical counterpart of ECE: it groups predictions into confidence bins and draws each bin's accuracy as a bar against the diagonal, so a perfectly calibrated model lies on the diagonal while bars below it reveal overconfidence and bars above reveal underconfidence.

In [ ]:
def reliability_data(probs, labels, n_bins=10):
    conf = probs.max(dim=1).values
    correct = (probs.argmax(dim=1) == labels).float()
    edges = torch.linspace(0.5, 1, n_bins + 1, device=probs.device)
    centers, accs, ece = [], [], 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = ((conf >= lo) if i == 0 else (conf > lo)) & (conf <= hi)
        centers.append(((lo + hi) / 2).item())
        if mask.any():
            a = correct[mask].mean().item(); c = conf[mask].mean().item()
            accs.append(a)
            ece += mask.float().mean().item() * abs(a - c)
        else:
            accs.append(np.nan)
    return dict(centers=centers, acc=accs, ece=ece)

def plot_reliability(ax, probs, labels, title):
    rd = reliability_data(probs, labels)
    w = rd["centers"][1] - rd["centers"][0]
    ax.bar(rd["centers"], rd["acc"], width=w, color="tab:blue", edgecolor="white",
           linewidth=0.5, label="accuracy")
    ax.plot([0.5, 1], [0.5, 1], "--", color="0.4", lw=0.8, label="perfect calibration")
    ax.set_title(f"{title} (ECE = {rd['ece']:.3f})")
    ax.set(xlabel="confidence", ylabel="accuracy", xlim=(0.5, 1), ylim=(0, 1),
           xticks=torch.linspace(0.5, 1, 3), yticks=torch.linspace(0, 1, 3))
    ax.legend(loc="upper left", frameon=False)

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5), sharey=True)
plot_reliability(axes[0], probs_adam, labels, "Adam")
plot_reliability(axes[1], probs_ivon, labels, "IVON@mean")
axes[1].set_ylabel("")
fig.tight_layout(); plt.show()

So far we've used LoRA adapter trained with IVON only at its posterior mean. IVON also learns a variance over every weight, so we can sample several weight, average their predictions, and watch the metrics improve as the ensemble grows. Let's load IVON's posterior, draw a few samples, and trace each metric against the number of samples.

In [ ]:
# A minimal implementation of the IVON optimizer to load the checkpoint
class IVON(torch.optim.Optimizer):
    def __init__(self, params, ess: float):
        if ess <= 0.0:
            raise ValueError(f"Invalid effective sample size: {ess}")
        super().__init__(params, {"ess": ess})

    def _iter_trainable_params(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p is not None and p.requires_grad:
                    yield group, p

    @contextmanager
    def sampled_params(self):
        samples = []
        with torch.no_grad():
            for group, p in self._iter_trainable_params():
                mean = p.detach().clone()
                p.add_(torch.randn_like(p) * (group["ess"] * self.state[p]["hess+wd"]).rsqrt())
                samples.append((p, mean))
        try:
            yield
        finally:
            with torch.no_grad():
                for p, mean in samples:
                    p.copy_(mean)

def ivon_with_hess(model, hess_path):
    trainable = [p for p in model.parameters() if p.requires_grad]
    flat = torch.load(hess_path, weights_only=True,map_location=device).to(device)
    opt = IVON(trainable, ess=ESS)
    off = 0
    for p in trainable:
        n = p.numel()
        opt.state[p]["hess+wd"] = flat[off:off + n].view_as(p).to(p.dtype)
        off += n
    assert off == flat.numel(), (off, flat.numel())
    return opt

@torch.inference_mode()
def collect_samples(model, loader, optimizer, num_samples, extra=4):
    pool = num_samples + extra
    model.eval()
    labels, sample_chunks = [], [[] for _ in range(pool)]
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels.append(batch["labels"])
        for s in range(pool):
            with optimizer.sampled_params():
                sample_chunks[s].append(torch.softmax(model(**batch), dim=-1))
    labels = torch.cat(labels)
    samples = torch.stack([torch.cat(c) for c in sample_chunks]).double()   # [pool, N, C]
    return samples, labels

def bma_curve_by_simulation(samples, labels, num_sims, max_samples=None):
    S = samples.shape[0]                                  # pool size
    max_samples = S if max_samples is None else max_samples
    rows = []
    for k in range(1, max_samples + 1):
        sim_metrics = []
        for _ in range(num_sims):
            idx = torch.randperm(S)[:k].to(samples.device)
            avg_probs = samples.index_select(0, idx).mean(dim=0)
            sim_metrics.append(calculate_metrics(avg_probs, labels))
        acc, nll, ece, brier = np.mean(sim_metrics, axis=0)   # average over simulations
        rows.append({"samples": k, "accuracy": acc, "nll": nll, "ece": ece, "brier": brier})
    return rows

N_SIMS = 100
ivon_opt = ivon_with_hess(model_ivon, "ckpts/ivon/hess.pt")

In [ ]:
samples_ivon, labels_bma = collect_samples(model_ivon, val_loader, ivon_opt, N_SAMPLES)
bma_probs = samples_ivon[:N_SAMPLES].mean(dim=0).float()   # BMA over N_SAMPLES draws
bma_curve = bma_curve_by_simulation(samples_ivon, labels_bma, num_sims=N_SIMS, max_samples=N_SAMPLES)
bma_curve_df = pd.DataFrame(bma_curve).set_index("samples")
print(f"IVON {N_SAMPLES} posterior samples, pool of {samples_ivon.shape[0]} ({N_SIMS} sims/point):")
print(bma_curve_df.round(4))

Now let's compare Adam, IVON at the posterior mean, and IVON with posterior sampling side by side:

In [ ]:
compare = pd.DataFrame([
    metrics_row("Adam", probs_adam, labels),
    metrics_row("IVON@mean", probs_ivon, labels),
    metrics_row(f"IVON@{N_SAMPLES} samples", bma_probs, labels_bma),
]).set_index("model")
print(compare.round(4))

titles = {"accuracy": "Accuracy", "nll": "NLL", "ece": "ECE", "brier": "Brier"}
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, metric in zip(axes, ["accuracy", "nll", "ece", "brier"]):
    ax.plot(bma_curve_df.index, bma_curve_df[metric], "o-", color="tab:red", ms=2.5, lw=1, label="IVON w/ sampling")
    ax.axhline(compare.loc["IVON@mean", metric], ls="--", color="tab:red", lw=1, label="IVON@mean")
    ax.axhline(compare.loc["Adam", metric], ls=":", color="tab:blue", lw=1, label="Adam")
    ax.set_title(titles[metric])
    ax.set_xlabel("# posterior samples")
    ax.grid(True, lw=0.4, alpha=0.4)
axes[0].yaxis.set_major_formatter("{x:.3f}")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

Let's compare their reliability diagrams too. Posterior sampling further improves IVON's calibration.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5, 2.5), sharey=True)
plot_reliability(axes[0], probs_ivon, labels, "IVON@mean")
plot_reliability(axes[1], bma_probs, labels_bma, f"IVON@{N_SAMPLES} Samples")
axes[1].set_ylabel("")
fig.tight_layout()
plt.show()

Finally, let's probe the models on two hand-written sentences, one ambiguous and one determinable from context, to see whether their confidence reflects the real uncertainty.

In [ ]:
WINO_PROMPT = ("Select one of the choices that answers the following question: "
               "{q} Choices: A. {o1}. B. {o2}. Answer:")

probe = [
    {"sentence": "I flipped a fair coin, which can land on heads or tails, and it came up _ .",
     "option1": "heads", "option2": "tails", "truth": None},
    {"sentence": ("Last week I ate cereal for breakfast every day and never ate pancakes, "
                  "so on Monday I ate _ for breakfast."),
     "option1": "cereal", "option2": "pancakes", "truth": "A"},
]
texts = [WINO_PROMPT.format(q=e["sentence"], o1=e["option1"], o2=e["option2"]) for e in probe]
print("Example prompt fed to the model:\n" + texts[0])

@torch.inference_mode()
def choice_probs(model, texts, optimizer=None, num_samples=0):
    model.eval()
    enc = tokenizer(texts, return_tensors="pt", padding=True)
    enc = {k: enc[k].to(device) for k in ("input_ids", "attention_mask")}
    if num_samples == 0:
        return torch.softmax(model(**enc), dim=-1).float()
    acc = None
    for _ in range(num_samples):
        with optimizer.sampled_params():
            p = torch.softmax(model(**enc), dim=-1).float()
        acc = p if acc is None else acc + p
    return acc / num_samples

with model_ivon.model.disable_adapter(): # un-finetuned base model (LoRA turned off)
    base_probs = choice_probs(model_ivon, texts)
preds = {
    "base (no FT)": base_probs,
    "Adam": choice_probs(model_adam, texts),
    "IVON@mean": choice_probs(model_ivon, texts),
    f"IVON@{N_SAMPLES} samples": choice_probs(model_ivon, texts, ivon_opt, N_SAMPLES),
}

for i, e in enumerate(probe):
    truth = "ambiguous" if e["truth"] is None else f"determinable -> {e['truth']}"
    print(f"\nQ: {e['sentence']}")
    print(f"   A = {e['option1']}   |   B = {e['option2']}   ({truth})")
    for name, p in preds.items():
        pA, pB = p[i, 0].item(), p[i, 1].item()
        print(f"   {name:16s}  P(A)={pA:.3f}  P(B)={pB:.3f}  -> {'A' if pA >= pB else 'B'}"
              f"  (conf {max(pA, pB):.3f})")

In [ ]:
import textwrap

palette = ["#999999", "#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]
names = list(preds.keys())
order = list(range(len(probe)))
prompt_prefix = WINO_PROMPT.split("{q}", maxsplit=1)[0]
xb, w = np.arange(2), 0.8 / len(preds)
fig, axes = plt.subplots(1, len(order), figsize=(2.8 * len(order), 3.2), squeeze=False, sharey=True)
for pos, j in enumerate(order):
    ax = axes[0, pos]
    for k, name in enumerate(names):
        ax.bar(xb + (k - (len(names) - 1) / 2) * w, preds[name][j].cpu().numpy(),
               width=w, label=name, color=palette[k], edgecolor="white", linewidth=0.4)
    ax.axhline(0.5, ls=":", color="0.5", lw=0.6)
    ax.set_xticks(xb)
    ax.set_xticklabels(["A", "B"])
    ax.set_ylim(0, 1)
    ax.set_title(textwrap.fill(texts[j].removeprefix(prompt_prefix), width=44))
axes[0, 0].set_ylabel("probability")
handles, labels = axes[0, 0].get_legend_handles_labels()
labels = [l.replace(f"BMA x{N_SAMPLES}", f"{N_SAMPLES} Samples") for l in labels]
fig.legend(handles, labels, loc="lower center", ncol=len(names), frameon=False)
fig.tight_layout(rect=[0, 0.08, 1, 0.94])
plt.show()